# 📘 Fincept Notebook — Inflation & Real Returns

**Economics · Beginner · ~12 min · Standard library only**

Inflation quietly erodes the value of money. This notebook starts from a Consumer Price Index (CPI) series, derives year-over-year inflation, converts nominal investment returns into real (inflation-adjusted) returns using the Fisher relation, shows how the purchasing power of idle cash decays, and uses the Rule of 72 to estimate doubling times.

**What you'll learn**
- How to compute year-over-year inflation from a CPI index
- The Fisher relation: turning nominal returns into real returns
- How inflation erodes the purchasing power of cash over time
- The Rule of 72 for estimating doubling (and halving) times


## 1. CPI index and year-over-year inflation

A **CPI** is an index that tracks the average price level of a basket of goods, anchored to a base year. The **year-over-year inflation rate** between two years is simply the percentage change in the index:

`inflation_t = CPI_t / CPI_{t-1} − 1`

Below is an embedded annual CPI series; we compute the inflation rate for each year.


In [ ]:
# Annual end-of-year CPI index (illustrative, base = 100.0 in the first year)
cpi = {
    2015: 100.0,
    2016: 101.3,
    2017: 103.4,
    2018: 105.9,
    2019: 107.8,
    2020: 109.2,
    2021: 114.4,
    2022: 123.6,
    2023: 128.7,
    2024: 132.6,
}

years = sorted(cpi)
print("Year-over-year inflation from the CPI series:")
print(f"{'Year':>6}{'CPI':>10}{'Inflation':>12}")
print('-' * 28)
inflation = {}
for prev, cur in zip(years, years[1:]):
    rate = cpi[cur] / cpi[prev] - 1
    inflation[cur] = rate
    print(f"{cur:>6}{cpi[cur]:>10.1f}{rate:>11.2%}")

# Average (compound annual) inflation over the whole span
n = years[-1] - years[0]
cagr = (cpi[years[-1]] / cpi[years[0]]) ** (1 / n) - 1
print('-' * 28)
print(f"\nCumulative price rise {years[0]}-{years[-1]}: {cpi[years[-1]] / cpi[years[0]] - 1:.1%}")
print(f"Average annual inflation (CAGR): {cagr:.2%}")


## 2. Nominal vs real returns (the Fisher relation)

A **nominal** return is the headline percentage your money grew. The **real** return strips out inflation to show the gain in actual purchasing power. The exact Fisher relation is:

`1 + real = (1 + nominal) / (1 + inflation)`  →  `real = (1 + nominal) / (1 + inflation) − 1`

A common shortcut, `real ≈ nominal − inflation`, is close but slightly overstates the real return; we show both.


In [ ]:
def real_return(nominal, inflation):
    """Exact Fisher relation: real return given a nominal return and inflation."""
    return (1 + nominal) / (1 + inflation) - 1

# A few investments and the inflation backdrop each faced
scenarios = [
    ('Savings account',  0.015, 0.032),
    ('Government bond',   0.035, 0.032),
    ('Balanced fund',     0.070, 0.045),
    ('Equity portfolio',  0.110, 0.045),
    ('High-inflation yr', 0.080, 0.090),
]

print("Nominal vs real returns (exact Fisher relation):")
print(f"{'Asset':<20}{'Nominal':>10}{'Inflation':>11}{'Real (exact)':>14}{'Approx':>9}")
print('-' * 64)
for name, nom, infl in scenarios:
    real = real_return(nom, infl)
    approx = nom - infl
    print(f"{name:<20}{nom:>10.2%}{infl:>11.2%}{real:>14.2%}{approx:>9.2%}")

print("\nNote: when nominal return < inflation, the real return is negative —")
print("your money grows in dollars but buys less than before.")


## 3. Purchasing power erosion of cash

Cash earning nothing loses real value every year inflation is positive. If you hold a fixed amount of money, its purchasing power in base-year terms is:

`real value_t = nominal amount × CPI_base / CPI_t`

We track $10,000 of idle cash across the same CPI series to see how much buying power it quietly loses.


In [ ]:
cpi = {
    2015: 100.0, 2016: 101.3, 2017: 103.4, 2018: 105.9, 2019: 107.8,
    2020: 109.2, 2021: 114.4, 2022: 123.6, 2023: 128.7, 2024: 132.6,
}

years = sorted(cpi)
base = years[0]
cash = 10000.0

print(f"Purchasing power of ${cash:,.0f} held as idle cash (in {base} dollars):")
print(f"{'Year':>6}{'CPI':>10}{'Real value':>14}{'Buying power lost':>20}")
print('-' * 50)
for y in years:
    real_value = cash * cpi[base] / cpi[y]
    lost = cash - real_value
    print(f"{y:>6}{cpi[y]:>10.1f}{real_value:>14,.2f}{lost:>20,.2f}")

final_real = cash * cpi[base] / cpi[years[-1]]
print('-' * 50)
print(f"\nOver {years[-1] - base} years, ${cash:,.0f} of cash kept only "
      f"${final_real:,.2f} of {base} purchasing power")
print(f"— a real loss of {(1 - final_real / cash):.1%}.")


## 4. The Rule of 72

The **Rule of 72** is a quick mental estimate: divide 72 by a percentage growth rate to approximate the years needed to double. It also works in reverse for inflation — dividing 72 by the inflation rate estimates how long until money's purchasing power halves. We compare the rule against the exact logarithmic answer.


In [ ]:
import math

def rule_of_72(rate_pct):
    return 72 / rate_pct

def exact_doubling(rate_pct):
    return math.log(2) / math.log(1 + rate_pct / 100)

print("Doubling time: Rule of 72 vs exact")
print(f"{'Rate':>6}{'Rule of 72':>14}{'Exact (yrs)':>14}{'Error':>10}")
print('-' * 44)
for rate in [2, 3, 4, 6, 8, 10, 12]:
    approx = rule_of_72(rate)
    exact = exact_doubling(rate)
    print(f"{rate:>5}%{approx:>14.1f}{exact:>14.1f}{approx - exact:>+10.2f}")

# Applied to inflation: how fast does purchasing power halve?
print("\nApplied to inflation — years for prices to double / money to halve:")
print(f"{'Inflation':>10}{'Rule of 72':>14}{'Exact (yrs)':>14}")
print('-' * 38)
for infl in [2, 3, 5, 7]:
    print(f"{infl:>9}%{rule_of_72(infl):>14.1f}{exact_doubling(infl):>14.1f}")


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
